# 2 · The 10 km target grid

Every dataset is aligned onto a single [`TargetGrid`](../api/grid.md#data4simplace.grid.TargetGrid), and every
cell carries one deterministic `SimplaceID` that is shared across the
weather, soil and management outputs. This notebook builds a grid, reads
its cell table, and demonstrates regridding. No external data required.

## Build a grid from a `GridConfig`

In [ ]:
from data4simplace.config import GridConfig
from data4simplace.grid import TargetGrid

grid_cfg = GridConfig(
    resolution_deg=0.1,
    min_lon=11.2, max_lon=14.8,
    min_lat=51.3, max_lat=53.6,
)
grid = TargetGrid.from_config(grid_cfg)
print('grid shape (n_lat, n_lon):', grid.shape)
print('n cells:', grid.shape[0] * grid.shape[1])

## The cell table and `SimplaceID`

`SimplaceID` is assigned in row-major order (north→south, west→east) and
is stable for a given bounding box + resolution.

In [ ]:
cells = grid.cell_table()
cells.head()

In [ ]:
print('columns:', list(cells.columns))
print('SimplaceID range:', cells.SimplaceID.min(), '->', cells.SimplaceID.max())

## Visualise the grid cells coloured by `SimplaceID`

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
sc = ax.scatter(cells.lon, cells.lat, c=cells.SimplaceID, s=14, cmap='viridis')
ax.set_xlabel('longitude'); ax.set_ylabel('latitude')
ax.set_title('Target grid cell centres (colour = SimplaceID)')
fig.colorbar(sc, ax=ax, label='SimplaceID')
plt.tight_layout()

## Regrid a finer field onto the target grid

`TargetGrid.regrid` bins a finer source field into each 10 km cell. Here
we synthesise a 0.02° field (5× finer) and aggregate it with the mean.

In [ ]:
import numpy as np
import xarray as xr

fine_res = 0.02
lon = np.arange(11.2 + fine_res / 2, 14.8, fine_res)
lat = np.arange(53.6 - fine_res / 2, 51.3, -fine_res)
field = np.sin(np.deg2rad(lat)[:, None] * 40) + np.cos(np.deg2rad(lon)[None, :] * 40)
da = xr.DataArray(field, dims=('lat', 'lon'), coords={'lat': lat, 'lon': lon}, name='demo')
print('source shape:', da.shape)

coarse = grid.regrid(da, method='mean')
print('regridded shape:', coarse.shape)

In [ ]:
fig, (a, b) = plt.subplots(1, 2, figsize=(10, 4))
da.plot(ax=a); a.set_title('source · 0.02 deg')
coarse.plot(ax=b); b.set_title('regridded · 0.1 deg (mean)')
plt.tight_layout()

## Why the shared `SimplaceID` matters

Because the grid is deterministic, the *same* cell gets the *same*
`SimplaceID` in the weather, soil and management files — that is what lets
SIMPLACE join them back together per point.